In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from ripser import ripser
from scipy.stats import linregress

In [ ]:
representations_path = "resnet_epoch1_vectorized_representations.pt"

representations = torch.load(
    representations_path,
    map_location="cpu"
)

print("Number of blocks:", len(representations))

for i, rep in enumerate(representations):
    print(f"Block {i+1}: {rep.shape}")

In [ ]:
random_seed = 42

generator = torch.Generator().manual_seed(random_seed)

random_order = torch.randperm(
    representations[0].shape[0],
    generator=generator
)

print("Total available points:", len(random_order))
print("First 10 indices:", random_order[:10])

In [ ]:
sample_sizes = np.array([
    50,
    100,
    150,
    200,
    250,
    300
])

print("Sample sizes:", sample_sizes)

In [ ]:
def lifespan_sum(diagram, alpha=1.0):

    finite_mask = np.isfinite(
        diagram[:, 1]
    )

    finite_pairs = diagram[
        finite_mask
    ]

    if len(finite_pairs) == 0:
        return 0.0

    lifespans = (
        finite_pairs[:, 1]
        - finite_pairs[:, 0]
    )

    return np.sum(
        lifespans ** alpha
    )

In [ ]:
block_index = 0

representation = representations[block_index]

e_values = []

for n in sample_sizes:

    indices_n = random_order[:n]

    sample = representation[
        indices_n
    ].numpy()

    print(f"Processing n = {n}")

    result = ripser(
        sample,
        maxdim=0
    )

    h0_diagram = result["dgms"][0]

    e_value = lifespan_sum(
        h0_diagram,
        alpha=1
    )

    e_values.append(e_value)

    print(
        f"E_1^0 = {e_value:.6f}"
    )

In [ ]:
e_values = np.array(e_values)

print("Sample sizes:")
print(sample_sizes)

print("\nE_1^0 values:")
print(e_values)

print("\nNaN:", np.isnan(e_values).any())
print("Inf:", np.isinf(e_values).any())

In [ ]:
log_n = np.log10(sample_sizes)
log_e = np.log10(e_values)

print("log10(n):")
print(log_n)

print("\nlog10(E_1^0):")
print(log_e)

In [ ]:
slope, intercept, r_value, p_value, std_err = linregress(
    log_n,
    log_e
)

print("Slope:", slope)
print("Intercept:", intercept)
print("R:", r_value)
print("R²:", r_value ** 2)

In [ ]:
alpha = 1.0
beta = slope

if beta < 1:
    phdim = alpha / (1 - beta)
else:
    phdim = np.nan

print("Beta:", beta)
print("PHdim:", phdim)

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    log_n,
    log_e,
    s=40,
    label="Measured values"
)

fit_line = slope * log_n + intercept

plt.plot(
    log_n,
    fit_line,
    label="Linear fit"
)

plt.xlabel("log10(n)")
plt.ylabel("log10(E₁⁰)")
plt.title("PHdim Scaling — ResNet Block 1")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("PHdim verification")
print("-------------------")

print("Number of sample sizes:", len(sample_sizes))
print("Beta:", beta)
print("R²:", r_value ** 2)
print("PHdim:", phdim)

print("\nFinite E values:",
      np.isfinite(e_values).all())

print("Finite PHdim:",
      np.isfinite(phdim))

In [ ]:
phdim_result = {
    "block": block_index + 1,
    "sample_sizes": torch.tensor(sample_sizes),
    "e_values": torch.tensor(e_values),
    "log_n": torch.tensor(log_n),
    "log_e": torch.tensor(log_e),
    "beta": float(beta),
    "r_squared": float(r_value ** 2),
    "phdim": float(phdim) if np.isfinite(phdim) else float("nan")
}

torch.save(
    phdim_result,
    "resnet_epoch1_block1_phdim_test.pt"
)

print(
    "Saved: "
    "resnet_epoch1_block1_phdim_test.pt"
)